<a href="https://colab.research.google.com/github/2anoFIAP/TreinoCP4-Nemec/blob/main/CP4Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
from scipy import stats


df = pd.read_csv('/content/dataset_pt_churrasco.csv')

df.head()

,quantidade_picanha,tempo_médio_de_boteco,n_de_amigos_com_nome_junior,índice_de_solvência_do_cooler,quantas_tias_fizeram_maionese,cor_do_terno_que_vai_sujar,vezes_que_cantou_evidências,nível_de_birita_dog,horario_chegada_carvao,temperatura_churrasqueiro,pt_no_churrasco
0,4.247241,0.620181,0,0.342246,1,estampado,1,6.321149,17,33.964253,1
1,7.704286,6.376256,1,0.472972,2,branco,0,28.723441,17,36.285942,1
2,6.391964,6.487621,0,0.411681,1,cinza,1,0.265899,12,38.780766,0
3,5.591951,7.649159,0,0.910248,3,cinza,1,29.934626,15,38.150410,1
4,2.936112,8.713096,1,0.453450,3,branco,1,20.304639,14,38.001331,1


In [10]:
#Hipotese 1 - Churrascos onde alguém cantou 'Evidências' pelo menos uma vez têm um nível médio de birita_dog significativamente maior do que churrascos sem nenhuma execução da obra do Chitãozinho & Xororó

cantou = df[df['vezes_que_cantou_evidências'] >= 1]['nível_de_birita_dog'].dropna()
nao_cantou = df[df['vezes_que_cantou_evidências'] == 0]['nível_de_birita_dog'].dropna()

print(f"Grupo 'Cantou' (n={len(cantou)}): média = {cantou.mean():.2f}")
print(f"Grupo 'Não cantou' (n={len(nao_cantou)}): média = {nao_cantou.mean():.2f}")
print()

alpha = 0.05

_, p_shapiro_cantou = stats.shapiro(cantou)
_, p_shapiro_nao_cantou = stats.shapiro(nao_cantou)
print(f"Shapiro 'Cantou' p-valor: {p_shapiro_cantou:.5f}")
print(f"Shapiro 'Não cantou' p-valor: {p_shapiro_nao_cantou:.5f}")

ambos_normais = (p_shapiro_cantou >= alpha) and (p_shapiro_nao_cantou >= alpha)

if ambos_normais:
  _,p_levene = stats.levene(cantou, nao_cantou)
  print(f"Levene p-valor: {p_levene:.5f}")

  if p_levene >= alpha:
    print("Dados normais e variacnias iguais (homocedasticidade): Teste t independente padrao")
    t_stat, p_value = stats.ttest_ind(cantou, nao_cantou, equal_var=True)

  else:
    stat, p_value = stats.ttest_ind(cantou, nao_cantou, equal_var=False)
    print("Dados normais e variacnias diferentes (heterocedasticidade): Teste t welch.")
else:
  print(" pelo menos um grupo nao e normal: teste de mann-witney.")
  stat, p_value = stats.mannwhitneyu(cantou, nao_cantou, alternative='two-sided')

print(f"\nEstatisticas: {stat:.4f} | p-valor: {p_value:.5f}")

if p_value < alpha:
    print("Decisão: Rejeitamos H0. O nível de birita_dog difere significativamente entre quem cantou Evidências e quem não cantou.")
else:
    print("Decisão: Não rejeitamos H0. Não há diferença estatisticamente significativa no nível de birita_dog entre os grupos.")

Grupo 'Cantou' (n=198): média = 14.37
Grupo 'Não cantou' (n=102): média = 15.54

Shapiro 'Cantou' p-valor: 0.00000
Shapiro 'Não cantou' p-valor: 0.00051
 pelo menos um grupo nao e normal: teste de mann-witney.

Estatisticas: 9298.0000 | p-valor: 0.26131
Decisão: Não rejeitamos H0. Não há diferença estatisticamente significativa no nível de birita_dog entre os grupos.


In [11]:
#Hipotese 2 - O horário médio de chegada do carvão é significativamente diferente de 12h (meio-dia)

dados = df['horario_chegada_carvao'].dropna()

alpha = 0.05

stat_shapiro, p_shapiro = stats.shapiro(dados)
print(f"Shapiro-Wilk p_valor: {p_shapiro:.5f}")

if p_shapiro > alpha:
  print("Dados normais: Teste t de 1 amostra")
  stat, p_valor  = stats.ttest_1samp(dados, popmean=12)
else:
  print("Dados nao-normais: teste de wilcoxon")
  diferenca = dados - 12
  stat, p_valor = stats.wilcoxon(diferenca)

print(f"\nEstatisticas: {stat:.4f} | p-valor: {p_valor:.5f}")

if p_valor < alpha:
    print("Decisão: Rejeitamos H0. O horário médio de chegada do carvão é significativamente diferente de 12h (meio-dia).")
else:
    print("Decisão: Não rejeitamos H0. Nao ha evidencias de que o carvao chegue em horario diferente de 12h")

Shapiro-Wilk p_valor: 0.00000
Dados nao-normais: teste de wilcoxon

Estatisticas: 6279.0000 | p-valor: 0.00000
Decisão: Rejeitamos H0. O horário médio de chegada do carvão é significativamente diferente de 12h (meio-dia).


In [12]:
#Hipotese 3 - Quem usa terno branco no churrasco tem uma temperatura corporal do churrasqueiro significativamente diferente de quem usa terno cinza.
branco = df[df['cor_do_terno_que_vai_sujar'] == 'branco']['temperatura_churrasqueiro'].dropna()
cinza = df[df['cor_do_terno_que_vai_sujar'] == 'cinza']['temperatura_churrasqueiro'].dropna()

print(f"Grupo 'Branco' (n={len(branco)}): média = {branco.mean():.2f}°C")
print(f"Grupo 'Cinza' (n={len(cinza)}): média = {cinza.mean():.2f}°C\n")

alpha = 0.05

_, p_shap_branco = stats.shapiro(branco)
_, p_shap_cinza = stats.shapiro(cinza)
print(f"Shapiro 'Branco' p-valor: {p_shap_branco:.5f}")
print(f"Shapiro 'Cinza' p-valor: {p_shap_cinza:.5f}")

ambos_normais = (p_shap_branco >= alpha) and (p_shap_cinza >= alpha)

if ambos_normais:
    _, p_levene = stats.levene(branco, cinza)
    print(f"Levene p-valor: {p_levene:.5f}")

    if p_levene >= alpha:
        print("-> Dados normais e variâncias iguais (homocedasticidade): Teste t independente padrão.")
        stat, p_valor = stats.ttest_ind(branco, cinza, equal_var=True)
    else:
        print("-> Dados normais e variâncias desiguais (heterocedasticidade): Teste t de Welch.")
        stat, p_valor = stats.ttest_ind(branco, cinza, equal_var=False)
else:
    print("-> Pelo menos um grupo não é normal: Teste de Mann-Whitney U.")
    stat, p_valor = stats.mannwhitneyu(branco, cinza, alternative='two-sided')

print(f"\nEstatística: {stat:.4f} | p-valor: {p_valor:.5f}")

if p_valor < alpha:
    print("Decisão: Rejeitamos H0. A temperatura do churrasqueiro difere significativamente entre quem usa branco e cinza.")
else:
    print("Decisão: Não rejeitamos H0. Não há diferença significativa na temperatura entre os grupos.")

Grupo 'Branco' (n=76): média = 36.91°C
Grupo 'Cinza' (n=72): média = 37.37°C

Shapiro 'Branco' p-valor: 0.48874
Shapiro 'Cinza' p-valor: 0.56840
Levene p-valor: 0.19255
-> Dados normais e variâncias iguais (homocedasticidade): Teste t independente padrão.

Estatística: -1.3905 | p-valor: 0.16650
Decisão: Não rejeitamos H0. Não há diferença significativa na temperatura entre os grupos.


In [13]:
#Hipotese 4 - A quantidade de picanha (em kg) influencia significativamente a chance de dar PT no churrasco.
pt = df[df['pt_no_churrasco'] == 1]['quantidade_picanha'].dropna()
nao_pt = df[df['pt_no_churrasco'] == 0]['quantidade_picanha'].dropna()

print(f"Grupo 'PT' (n={len(pt)}): média = {pt.mean():.2f} kg")
print(f"Grupo 'Não-PT' (n={len(nao_pt)}): média = {nao_pt.mean():.2f} kg\n")

_, p_shap_pt = stats.shapiro(pt)
_, p_shap_nao_pt = stats.shapiro(nao_pt)
print(f"Shapiro 'PT' p-valor: {p_shap_pt:.5f}")
print(f"Shapiro 'Não-PT' p-valor: {p_shap_nao_pt:.5f}")

ambos_normais = (p_shap_pt >= alpha) and (p_shap_nao_pt >= alpha)

if ambos_normais:
    _, p_levene = stats.levene(pt, nao_pt)
    print(f"Levene p-valor: {p_levene:.5f}")

    if p_levene >= alpha:
        print("-> Dados normais e variâncias iguais (homocedasticidade): Teste t independente padrão.")
        stat, p_valor = stats.ttest_ind(pt, nao_pt, equal_var=True)
    else:
        print("-> Dados normais e variâncias desiguais (heterocedasticidade): Teste t de Welch.")
        stat, p_valor = stats.ttest_ind(pt, nao_pt, equal_var=False)
else:
    print("-> Pelo menos um grupo não é normal: Teste de Mann-Whitney U.")
    stat, p_valor = stats.mannwhitneyu(pt, nao_pt, alternative='two-sided')

print(f"\nEstatística: {stat:.4f} | p-valor: {p_valor:.5f}")

if p_valor < alpha:
    print("Decisão: Rejeitamos H0. A quantidade de picanha difere significativamente entre quem deu PT e quem não deu.")
else:
    print("Decisão: Não rejeitamos H0. Não há diferença significativa na quantidade de picanha entre os grupos.")

Grupo 'PT' (n=279): média = 4.90 kg
Grupo 'Não-PT' (n=21): média = 5.91 kg

Shapiro 'PT' p-valor: 0.00000
Shapiro 'Não-PT' p-valor: 0.76712
-> Pelo menos um grupo não é normal: Teste de Mann-Whitney U.

Estatística: 1960.0000 | p-valor: 0.01148
Decisão: Rejeitamos H0. A quantidade de picanha difere significativamente entre quem deu PT e quem não deu.


In [14]:
#Hipotese 5 (GOAT) - Quanto mais tias fizeram maionese, MAIOR o índice de solvência do cooler — porque no fundo todo mundo sabe que a função real das tias no churrasco não é contribuir com a salada, é garantir que a cerveja não acabe

poucas_tias = df[df['quantas_tias_fizeram_maionese'] <= 1]['índice_de_solvência_do_cooler'].dropna()
muitas_tias = df[df['quantas_tias_fizeram_maionese'] >= 2]['índice_de_solvência_do_cooler'].dropna()

_, p_shap_poucas = stats.shapiro(poucas_tias)
_, p_shap_muitas = stats.shapiro(muitas_tias)
print(f"Shapiro 'Poucas tias' p-valor: {p_shap_poucas:.5f}")
print(f"Shapiro 'Muitas tias' p-valor: {p_shap_muitas:.5f}")


ambas_normais = (p_shap_poucas>= alpha) and (p_shap_muitas >= alpha)

if ambas_normais:
  _, p_levene = stats.levene(poucas_tias, muitas_tias)
  print(f"Levene p-valor: {p_levene:.5f}")

  if p_levene >= alpha:
    print("Dados normais e variâncias iguais (homocedasticidade): Teste t independente padrão.")
    stat, p_valor = stats.ttest_ind(poucas_tias, muitas_tias, equal_var=True)
  else:
    print("Dados normais e variâncias desiguais (heterocedasticidade): Teste t de Welch.")
    stat, p_valor = stats.ttest_ind(poucas_tias, muitas_tias, equal_var=False)
else:
  print("Pelo menos um grupo não é normal: Teste de Mann-Whitney U.")
  stat, p_valor = stats.mannwhitneyu(poucas_tias, muitas_tias, alternative='two-sided')

print(f"\nEstatística: {stat:.4f} | p-valor: {p_valor:.5f}")

if p_valor < alpha:
    print("Decisão: Rejeitamos H0. CONFIRMADO: tia de maionese é, na verdade, tia de logística etílica. 🍺")
else:
    print("Decisão: Não rejeitamos H0. As tias realmente só estão fazendo maionese. Inocentes. 🥗")

Shapiro 'Poucas tias' p-valor: 0.00001
Shapiro 'Muitas tias' p-valor: 0.00117
Pelo menos um grupo não é normal: Teste de Mann-Whitney U.

Estatística: 11277.0000 | p-valor: 0.94582
Decisão: Não rejeitamos H0. As tias realmente só estão fazendo maionese. Inocentes. 🥗
